In [1]:
import pandas as pd

def analizar_indices(fichero):
    try:
        # Leer el CSV
        df = pd.read_csv(fichero)
        
        # Extraer los valores como conjuntos (sets) para buscar diferencias
        set_none = set(df['n0'].dropna())
        set_spherical = set(df['c0'].dropna())
        
        # 1. Índices que están en NONE pero NO en SPHERICAL (Puntos perdidos por la optimización)
        solo_en_none = set_none - set_spherical
        
        # 2. Índices que están en SPHERICAL pero NO en NONE (Puntos extraños/falsos positivos)
        solo_en_spherical = set_spherical - set_none
        
        print(f"--- Análisis de {fichero} ---")
        print(f"Total registros en 'none': {len(set_none)}")
        print(f"Total registros en 'spherical': {len(set_spherical)}\n")
        
        print(f"Puntos perdidos (están en 'none' pero no en 'spherical'):")
        if solo_en_none:
            print(sorted(list(solo_en_none)))
        else:
            print("Ninguno")
            
        print(f"\nFalsos positivos (están en 'spherical' pero no en 'none'):")
        if solo_en_spherical:
            print(sorted(list(solo_en_spherical)))
        else:
            print("Ninguno")

        # También podemos ver en qué filas específicas fallan
        print("\n--- Discrepancias por número de fila (índice 0) ---")
        discrepancias = df[df['none'] != df['spherical']]
        if not discrepancias.empty:
            print(discrepancias)
        else:
            print("No hay discrepancias fila a fila.")

    except FileNotFoundError:
        print(f"Error: No se encontró el archivo '{fichero}'")
    except Exception as e:
        print(f"Ocurrió un error: {e}")

analizar_indices('indices.csv')

--- Análisis de indices.csv ---
Total registros en 'none': 116
Total registros en 'spherical': 70

Puntos perdidos (están en 'none' pero no en 'spherical'):
[10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 30.0, 31.0, 33.0, 34.0, 35.0, 36.0, 37.0, 47.0, 48.0, 49.0, 58.0, 59.0, 60.0, 61.0, 62.0, 63.0, 64.0, 65.0, 66.0, 67.0, 68.0, 69.0, 70.0, 73.0, 360.0, 361.0, 363.0, 364.0]

Falsos positivos (están en 'spherical' pero no en 'none'):
Ninguno

--- Discrepancias por número de fila (índice 0) ---
Ocurrió un error: 'none'


### Comprobar distancias de los puntos perdidos

In [20]:
import numpy as np

# Hoja 38
center = np.array([-32.20, -31.84, 52.41])
q = np.array([-34.87, -38.28, 58.18])  # query de búsqueda
radius_cube = 15.0
radius = radius_cube * np.sqrt(3)  # esfera envolvente del cubo

# Puntos que faltan
points = {
    360: np.array([-26.391, -25.720, 54.257]),
    361: np.array([-35.724, -24.711, 53.486]),
    363: np.array([-36.312, -23.717, 52.086]),
    364: np.array([-37.350, -28.530, 52.129]),
}

# ── Rango K0 (ángulo azimutal) ──────────────────────────────────────────────

def normalize_angle(a):
    out = np.fmod(a, 2 * np.pi)
    return out + 2 * np.pi if out < 0 else out

# dxy: distancia en XY del centro de la hoja al query
dq = q - center
dxy = np.sqrt(dq[0]**2 + dq[1]**2)

phiQ     = normalize_angle(np.arctan2(dq[1], dq[0]))
deltaPhi = np.arcsin(np.clip(radius / dxy, 0.0, 1.0))
kMinRaw  = phiQ - deltaPhi
kMaxRaw  = phiQ + deltaPhi

print("=" * 65)
print(f"  Query relativo al centro:  dx={dq[0]:.4f}  dy={dq[1]:.4f}  dxy={dxy:.4f}")
print(f"  radius (cubo envuelto):    {radius:.4f}  (semilado={radius_cube})")
print(f"  phiQ     = {np.degrees(phiQ):.4f}°  ({phiQ:.6f} rad)")
print(f"  deltaPhi = {np.degrees(deltaPhi):.4f}°  ({deltaPhi:.6f} rad)")
print(f"  Rango K0 = [{np.degrees(kMinRaw):.4f}°, {np.degrees(kMaxRaw):.4f}°]")
wrap = kMinRaw < 0 or kMaxRaw >= 2 * np.pi
print(f"  Wrap (devuelve full):      {wrap}")
print("=" * 65)

# ── Tabla de puntos ─────────────────────────────────────────────────────────

print(f"\n{'ID':>5} {'dx_q':>8} {'dy_q':>8} {'dz_q':>8} "
      f"{'rxy_q':>8} {'r3d_q':>8} "
      f"{'phi_c':>10} {'phi_q':>10} "
      f"{'in_cube':>8} {'in_K0':>8}")
print("-" * 90)

for pid, p in points.items():
    # relativo al query (para test cubo)
    drel_q = p - q
    rxy_q  = np.sqrt(drel_q[0]**2 + drel_q[1]**2)
    r3d_q  = np.linalg.norm(drel_q)

    # dentro del cubo?
    in_cube = all(abs(drel_q[:3]) <= radius_cube)

    # ángulo del punto relativo al centro de la hoja
    drel_c  = p - center
    phi_c   = normalize_angle(np.arctan2(drel_c[1], drel_c[0]))

    # ángulo del punto relativo al query
    phi_q_val = normalize_angle(np.arctan2(drel_q[1], drel_q[0]))

    # ¿está dentro del rango K0?
    in_K0 = kMinRaw <= phi_c <= kMaxRaw

    print(f"{pid:>5} {drel_q[0]:>8.3f} {drel_q[1]:>8.3f} {drel_q[2]:>8.3f} "
          f"{rxy_q:>8.3f} {r3d_q:>8.3f} "
          f"{np.degrees(phi_c):>10.3f}° {np.degrees(phi_q_val):>10.3f}° "
          f"{'YES' if in_cube else 'NO':>8} {'YES' if in_K0 else 'NO':>8}")

print()
print("Nota: phi_c = ángulo del punto respecto al CENTRO de la hoja (el que usa K0)")
print("      phi_q = ángulo del punto respecto al QUERY (referencia visual)")
print("      in_K0 = phi_c dentro del rango [kMinRaw, kMaxRaw]")

  Query relativo al centro:  dx=-2.6700  dy=-6.4400  dxy=6.9715
  radius (cubo envuelto):    25.9808  (semilado=15.0)
  phiQ     = 247.4813°  (4.319363 rad)
  deltaPhi = 90.0000°  (1.570796 rad)
  Rango K0 = [157.4813°, 337.4813°]
  Wrap (devuelve full):      False

   ID     dx_q     dy_q     dz_q    rxy_q    r3d_q      phi_c      phi_q  in_cube    in_K0
------------------------------------------------------------------------------------------
  360    8.479   12.560   -3.923   15.154   15.654     46.493°     55.978°      YES       NO
  361   -0.854   13.569   -4.694   13.596   14.383    116.304°     93.601°      YES       NO
  363   -1.442   14.563   -6.094   14.634   15.852    116.849°     95.655°      YES       NO
  364   -2.480    9.750   -6.051   10.060   11.740    147.270°    104.271°      YES       NO

Nota: phi_c = ángulo del punto respecto al CENTRO de la hoja (el que usa K0)
      phi_q = ángulo del punto respecto al QUERY (referencia visual)
      in_K0 = phi_c dentro del r

In [1]:
import pandas as pd

df = pd.read_csv('logs/funcv1Lille.log')

df['pruned_pct'] = (df['total'] - df['count']) / df['total'] * 100

summaryF1 = (
    df.groupby(['mode', 'key', 'kernel', 'radius'])['pruned_pct']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .round(2)
    .reset_index()
)

summaryF1.columns = ['mode', 'key', 'kernel', 'radius', 'mean_%', 'median_%', 'min_%', 'max_%', 'n_leaves']

In [4]:
summary

,mode,key,mean_%,median_%,min_%,max_%,n_leaves
0,cylindrical,0,26.54,28.57,0.0,66.67,16
1,cylindrical,1,3.33,0.00,0.0,36.36,16
2,cylindrical,2,0.00,0.00,0.0,0.00,16
3,spherical,0,26.54,28.57,0.0,66.67,16
4,spherical,1,0.00,0.00,0.0,0.00,16
5,spherical,2,4.40,0.00,0.0,36.36,16


In [6]:
summaryL

,mode,key,mean_%,median_%,min_%,max_%,n_leaves
0,cylindrical,0,45.95,48.48,0.0,100.0,49806
1,cylindrical,1,0.00,0.00,0.0,0.0,49806
2,cylindrical,2,0.00,0.00,0.0,0.0,49806
3,spherical,0,45.95,48.48,0.0,100.0,49806
4,spherical,1,0.00,0.00,0.0,0.0,49806
5,spherical,2,0.00,0.00,0.0,0.0,49806


#### Modificando radio
Después de aplicar las optimizaciones del radio elegido a K1, K2

 Dataset Lille_0

In [2]:
summaryF1

,mode,key,kernel,radius,mean_%,median_%,min_%,max_%,n_leaves
0,cylindrical,0,cube,0.5,44.54,48.39,0.0,100.00,817
1,cylindrical,0,cube,1.0,23.10,0.00,0.0,100.00,2653
2,cylindrical,0,cube,2.0,8.47,0.00,0.0,100.00,11190
3,cylindrical,0,cube,3.0,6.35,0.00,0.0,100.00,26348
4,cylindrical,0,cube,5.0,2.27,0.00,0.0,100.00,90728
5,cylindrical,0,sphere,0.5,66.75,75.81,0.0,100.00,817
6,cylindrical,0,sphere,1.0,41.33,44.74,0.0,100.00,2653
7,cylindrical,0,sphere,2.0,31.64,0.00,0.0,100.00,11190
8,cylindrical,0,sphere,3.0,27.93,0.00,0.0,100.00,26348
9,cylindrical,0,sphere,5.0,23.71,0.00,0.0,100.00,90728


### Deprecated (antes de arreglar rangos de KO)

Dataset Paris_Luxembourg_6

In [14]:
summary5

,mode,key,kernel,radius,mean_%,median_%,min_%,max_%,n_leaves
0,cylindrical,0,cube,0.5,22.88,0.0,0.0,100.00,599
1,cylindrical,0,cube,1.0,20.88,0.0,0.0,100.00,2273
2,cylindrical,0,cube,2.0,20.25,0.0,0.0,100.00,7439
3,cylindrical,0,cube,3.0,21.33,0.0,0.0,100.00,14622
4,cylindrical,0,cube,5.0,24.30,0.0,0.0,100.00,37572
5,cylindrical,0,sphere,0.5,26.07,0.0,0.0,100.00,599
6,cylindrical,0,sphere,1.0,22.82,0.0,0.0,100.00,2273
7,cylindrical,0,sphere,2.0,22.26,0.0,0.0,100.00,7439
8,cylindrical,0,sphere,3.0,23.92,0.0,0.0,100.00,14622
9,cylindrical,0,sphere,5.0,26.92,0.0,0.0,100.00,37572


Dataset Lille_0

In [17]:
summary6

,mode,key,kernel,radius,mean_%,median_%,min_%,max_%,n_leaves
0,cylindrical,0,cube,0.5,0.00,0.00,0.00,0.00,11
1,cylindrical,0,cube,1.0,33.58,0.00,0.00,100.00,73
2,cylindrical,0,cube,2.0,26.12,0.00,0.00,100.00,333
3,cylindrical,0,cube,3.0,27.59,0.00,0.00,100.00,768
4,cylindrical,0,cube,5.0,33.47,15.79,0.00,100.00,2567
5,cylindrical,0,sphere,0.5,19.94,0.00,0.00,92.19,11
6,cylindrical,0,sphere,1.0,43.84,0.00,0.00,100.00,73
7,cylindrical,0,sphere,2.0,50.79,56.25,0.00,100.00,333
8,cylindrical,0,sphere,3.0,38.93,33.07,0.00,100.00,768
9,cylindrical,0,sphere,5.0,41.18,33.33,0.00,100.00,2567
